# This notebook is for checking the integrity of the feature data downloaded from NERSC

In [52]:
import pathlib

In [53]:
feature_dir = pathlib.Path("/home/dileep/Documents/Work/DATA/biolog-annotations-1416")

In [54]:
feature_files = list(feature_dir.glob("*.tsv"))
feature_files, len(feature_files)

([PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/cluster-level-90.0.counts.tsv'),
  PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/cluster-level-30.0.counts.tsv'),
  PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/rast-annotations.tsv'),
  PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/eggnog-annotations-seed-orthologs.tsv'),
  PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/uniref90-annotations.tsv'),
  PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/eggnog-annotations-kegg-ids.tsv'),
  PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/cluster-level-50.0.counts.tsv'),
  PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/uniprot-trembl-annotations.tsv'),
  PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/uniref30-annotations.tsv'),
  PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/cluster-lev

There are 14 feature files

In [55]:
import subprocess

In [56]:
# Counting lines in the files
def count_lines_wc(filename):
    result = subprocess.run(['wc', '-l', str(filename)], text=True, capture_output=True)
    return int(result.stdout.split()[0])

In [57]:
counts = dict()
for feature_file in feature_files:
    counts[feature_file.stem] = count_lines_wc(feature_file) - 1
counts

{'cluster-level-90.0.counts': 1416,
 'cluster-level-30.0.counts': 1416,
 'rast-annotations': 1208,
 'eggnog-annotations-seed-orthologs': 1416,
 'uniref90-annotations': 1416,
 'eggnog-annotations-kegg-ids': 1416,
 'cluster-level-50.0.counts': 1416,
 'uniprot-trembl-annotations': 1416,
 'uniref30-annotations': 1416,
 'cluster-level-70.0.counts': 1416,
 'kofam-modules': 1416,
 'uniref50-annotations': 1416,
 'uniref70-annotations': 1416,
 'kofam-annotations': 1416}

In [58]:
len(set(counts.values()))

2

It looks like the RAST annoations are missing a couple of hundred genomes. Let us look at which ones are missing

In [59]:
# Get the first column of the file quickly using awk
def get_first_col(filename):
    cmd_prefix = "awk '{print $1 }'"
    cmd = cmd_prefix + " " + str(filename)
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    raw_col = result.stdout.split()[1:]
    cols = [x.strip().split("?")[-1].removesuffix(".RAST").removesuffix(".fna") for x in raw_col]
    return cols

In [60]:
feature_files[0], feature_files[3]

(PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/cluster-level-90.0.counts.tsv'),
 PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/eggnog-annotations-seed-orthologs.tsv'))

In [61]:
set(get_first_col(feature_files[0])) - set(get_first_col(feature_files[3]))

set()

First, let use check if all the feature files except rast have the same rows

In [62]:
from functools import reduce

genomes = {}
for feature_file in feature_files:
    if "rast" in feature_file.stem:
        continue
    genomes[feature_file.stem] = set(get_first_col(feature_file))

intersection = reduce(lambda a, b: a.intersection(b), genomes.values())
union = reduce(lambda a, b: a.union(b), genomes.values())

In [63]:
len(intersection), len(union), print("Expected: 1416")

Expected: 1416


(1127, 1127, None)

There is inconsistency -> There are duplicate genomes

In [64]:
print(feature_files[2])
if len(get_first_col(feature_files[2])) != len(set(get_first_col(feature_files[2]))):
    print("There are duplicates in the rast file")
else:
    print("No duplicates in the rast file")

/home/dileep/Documents/Work/DATA/biolog-annotations-1416/rast-annotations.tsv
There are duplicates in the rast file


In [65]:
from collections import Counter

c = Counter(get_first_col(feature_files[0]))
c.most_common()

[('GCF_001424325.1', 3),
 ('GCF_001422685.1', 3),
 ('GCF_001423585.1', 3),
 ('GCF_001424365.1', 3),
 ('GCF_001421405.1', 3),
 ('GCF_001422185.1', 3),
 ('GCF_916855635.1', 3),
 ('GCF_001421745.1', 3),
 ('GCF_001423985.1', 3),
 ('GCF_001422985.1', 3),
 ('GCF_001421805.1', 3),
 ('GCF_001421505.1', 3),
 ('GCF_920984665.1', 3),
 ('GCF_001422365.1', 3),
 ('GCF_001424225.1', 3),
 ('GCF_001423905.1', 3),
 ('GCF_001423845.1', 3),
 ('GCF_001421565.1', 3),
 ('GCF_001423295.1', 3),
 ('GCF_001421965.1', 3),
 ('GCF_920939485.1', 3),
 ('GCF_001423445.1', 3),
 ('GCF_001422865.1', 3),
 ('GCF_001423685.1', 3),
 ('GCF_001423725.1', 3),
 ('GCF_001423665.1', 3),
 ('GCF_001423045.1', 3),
 ('GCF_001425385.1', 3),
 ('GCF_001422445.1', 3),
 ('GCF_001421415.1', 3),
 ('GCF_001423885.1', 3),
 ('GCF_001421155.1', 3),
 ('GCF_001426225.1', 3),
 ('GCF_920939475.1', 3),
 ('GCF_001421285.1', 3),
 ('GCF_001424665.1', 3),
 ('GCF_003258395.1', 3),
 ('GCF_001422245.1', 3),
 ('GCF_001424195.1', 3),
 ('GCF_001422045.1', 3),


In [66]:
# Number of duplicated genomes
duplicates = {k: v for k, v in c.items() if v > 1}
len(duplicates)

197

In [67]:
394 + 103

497

In [68]:
for genome, count in c.most_common():
    if count > 1 and "GCF_" not in genome:
        print(genome, count)

This means that only the leaf genomes are duplicated

In [69]:
# How many leaf genomes are there?
len([x for x in intersection if "GCF_" in x])

208

There are 394 unique leaf genomes

---

In [70]:
394 - 186

208

## What is missing in RAST?

In [71]:
rast_feature_file = [f for f in feature_files if "rast" in f.stem][0]
uniref_feature_file = [f for f in feature_files if "uniref" in f.stem][0]
rast_feature_file, uniref_feature_file

(PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/rast-annotations.tsv'),
 PosixPath('/home/dileep/Documents/Work/DATA/biolog-annotations-1416/uniref90-annotations.tsv'))

In [72]:
rast_missing_genomes = union - set(get_first_col(rast_feature_file))
rast_missing_genomes, len(rast_missing_genomes)

({'GCF_001421315.1',
  'GCF_001421435.1',
  'GCF_001423085.1',
  'GCF_001423325.1',
  'GCF_001423605.1',
  'GCF_001424065.1',
  'GCF_001424185.1',
  'GCF_001424345.1',
  'GCF_001424505.1',
  'GCF_001425545.1',
  'GCF_001426065.1'},
 11)

In [73]:
leaf_count = 0
not_leaf = []
for genome in rast_missing_genomes:
    if 'GCF_' in genome:
        leaf_count += 1
    else:
        not_leaf.append(genome)
if leaf_count == len(rast_missing_genomes):
    print("All genomes in rast_missing_genomes are from the leaf dataset")
else:
    print("Some genomes in rast_missing_genomes are not from the leaf dataset")
    print(not_leaf)

All genomes in rast_missing_genomes are from the leaf dataset


Let us compare this with our phenotype list and see if any of the missing genomes are in the phenotype list

In [74]:
phenotype_dir = pathlib.Path("../../data/raw/biolog/phenotypes/")
leaf_phenotype_file = phenotype_dir / "p_LEAF_phenotypes.tsv"
ch_phenotype_file = phenotype_dir / "p_CH_phenotypes.tsv"
pmi_phenotype_file = phenotype_dir / "p_PMI_phenotypes.tsv"

In [75]:
leaf_genomes = get_first_col(leaf_phenotype_file)
ch_genomes = [g.removeprefix('g') for g in get_first_col(ch_phenotype_file)]
pmi_genomes = get_first_col(pmi_phenotype_file)
len(leaf_genomes), len(ch_genomes), len(pmi_genomes)

(206, 362, 55)

In [76]:
len(set(rast_missing_genomes) & set(leaf_genomes))

11

In [77]:
rast_genomes = get_first_col(rast_feature_file)

In [78]:
uniref_genomes = get_first_col(uniref_feature_file)

In [79]:
len(set(rast_genomes) & set(leaf_genomes)), len(leaf_genomes)

(195, 206)

In [80]:
leaf_missing = set(leaf_genomes) - set(rast_genomes)
ch_missing = set(ch_genomes) - set(rast_genomes)
pmi_missing = set(pmi_genomes) - set(rast_genomes)
len(leaf_missing), len(ch_missing), len(pmi_missing)

(11, 0, 0)

In [81]:
[g for g in rast_genomes if 'GCF_' in g]

['GCF_001424285.1',
 'GCF_001423985.1',
 'GCF_001421585.1',
 'GCF_920984665.1',
 'GCF_916858655.1',
 'GCF_916855635.1',
 'GCF_001421805.1',
 'GCF_001421535.1',
 'GCF_001425965.1',
 'GCF_001421405.1',
 'GCF_001422925.1',
 'GCF_001421665.1',
 'GCF_001421165.1',
 'GCF_916855635.1',
 'GCF_001421945.1',
 'GCF_001424565.1',
 'GCF_001423925.1',
 'GCF_001422615.1',
 'GCF_001421845.1',
 'GCF_920984675.1',
 'GCF_001423865.2',
 'GCF_001421715.1',
 'GCF_920984665.1',
 'GCF_916858655.1',
 'GCF_001425605.1',
 'GCF_001422065.1',
 'GCF_001421825.1',
 'GCF_001421425.1',
 'GCF_001424545.1',
 'GCF_920939475.1',
 'GCF_001423905.1',
 'GCF_001421915.1',
 'GCF_001424305.1',
 'GCF_001424445.1',
 'GCF_001423725.1',
 'GCF_001425345.1',
 'GCF_001423825.1',
 'GCF_001421285.1',
 'GCF_001421535.1',
 'GCF_001423185.1',
 'GCF_001423585.1',
 'GCF_001423785.1',
 'GCF_916862475.1',
 'GCF_001424755.1',
 'GCF_001422495.1',
 'GCF_920984705.1',
 'GCF_001423265.1',
 'GCF_001422665.1',
 'GCF_001421565.1',
 'GCF_001422605.1',


In [82]:
'GCF_001421165.1' in rast_genomes

True

In [83]:
leaf_missing

{'GCF_001421315.1',
 'GCF_001421435.1',
 'GCF_001423085.1',
 'GCF_001423325.1',
 'GCF_001423605.1',
 'GCF_001424065.1',
 'GCF_001424185.1',
 'GCF_001424345.1',
 'GCF_001424505.1',
 'GCF_001425545.1',
 'GCF_001426065.1'}

In [85]:
leaf_missing = set(leaf_genomes) - set(uniref_genomes)
ch_missing = set(ch_genomes) - set(uniref_genomes)
pmi_missing = set(pmi_genomes) - set(uniref_genomes)
len(leaf_missing), len(ch_missing), len(pmi_missing)

(0, 0, 0)

Conclusion:
Genomes are only missing from the RAST feature matrix.

In [86]:
[g for g in union if g.startswith("g")]

[]

In [87]:
[g for g in rast_genomes if g.startswith("g")]

[]